# Qwen3-0.6B × stock vLLM FlashAttention backend (reference)

이 노트북은 **vLLM 내장 `AttentionBackendEnum.FLASH_ATTN`** 만 사용하는 reference 다.
옆 자리의 `qwen3_flashattn_attention.ipynb` (커스텀 플러그인 = `MyFlashAttnBackend`)
가 에러를 낼 때, **stock backend 자체는 정상으로 도는가** 를 먼저 확인하기 위한 베이스라인.

차이 요약:

| | reference (이 노트북) | educational (옆 노트북) |
|---|---|---|
| backend slot | `FLASH_ATTN` (vLLM 빌트인) | `CUSTOM` (entry point 로 register) |
| 구현체 | `vllm.v1.attention.backends.flash_attn.FlashAttentionBackend` | `flash_attn_attention_backend:MyFlashAttnBackend` |
| plugin 의존 | 없음 (`pip install -e .` 불필요) | 있음 |
| 관찰 로그 | 없음 (stock 은 educational 로그 없음) | `MyFlashAttnImpl.forward fired ...` |

**용도**: 환경 (vllm + Qwen3-0.6B) 의 동작 여부를 분리 검증.
여기서 PASS 하면 → 문제는 **커스텀 backend 코드** 안에 있다고 좁힐 수 있다.
여기서 FAIL 하면 → 환경 (vLLM 버전 / GPU / driver) 자체가 문제.

## 1. 환경 확인

vLLM 0.19.x + CUDA GPU.

In [ ]:
import torch, vllm

print('cuda      :', torch.cuda.is_available())
print('gpu       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm      :', vllm.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 stock FlashAttentionBackend 시그니처가 다를 수 있음.'
)

## 2. Qwen3-0.6B 구조 확인

FA2 제약: `head_dim % 8 == 0 and head_dim <= 256`, `block_size % 16 == 0`.
Qwen3-0.6B 는 `head_dim=128`, vLLM 기본 `block_size=16` 이므로 둘 다 OK.

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden  :', cfg.hidden_size)
print('Q heads :', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers  :', cfg.num_hidden_layers)

assert hd % 8 == 0 and hd <= 256, 'FA2 는 head_dim % 8 == 0 and head_dim <= 256 필요'

## 3. Backend slot 확인

`AttentionBackendEnum.FLASH_ATTN` 이 vLLM 빌트인 `FlashAttentionBackend` 클래스 경로로
매핑되어 있는지 확인. 이 노트북은 **CUSTOM 슬롯을 건드리지 않는다** — 따라서 옆 노트북의
플러그인이 함께 등록되어 있어도 무관.

In [ ]:
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.FLASH_ATTN.get_path()
print('FLASH_ATTN slot ->', path)
assert 'FlashAttentionBackend' in path, f'예상 외 경로: {path}'

## 4. LLM 로드

`attention_backend=AttentionBackendEnum.FLASH_ATTN` 한 줄로 stock FA2 가 붙는다.
옆 노트북과 동일한 (`max_num_batched_tokens=64`) 설정으로 chunked prefill 도 트리거.

In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.FLASH_ATTN,
    enforce_eager=True,
    max_num_seqs=4,
    max_model_len=2048,
    max_num_batched_tokens=64,
)

## 5. Generate

옆 노트북과 동일한 prompt 셋. 긴 prompt 가 chunked prefill 로 쪼개진다.

In [ ]:
long_prompt = (
    'In the long history of artificial intelligence research, from the early '
    'symbolic AI of the 1950s through the neural network revival of the 1980s, '
    'the deep learning breakthroughs of the 2010s, and the transformer-based '
    'large language models of the 2020s, one theme has remained constant: '
    'the answer is'
)

prompts = [
    'The capital of France is',
    long_prompt,
    'Shakespeare wrote the play',
    'Python was created by',
]
out = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=16))
for i, o in enumerate(out):
    print(f'[{i}] (prompt {len(prompts[i])} chars) {o.outputs[0].text[:80]}')

## 6. PASS 기준

- 위 셀이 traceback 없이 끝나고
- 4개 prompt 모두 비어 있지 않은 텍스트가 출력되면

→ stock FlashAttention backend × Qwen3-0.6B × 이 환경 조합은 **건강하다**. 옆 노트북
(`qwen3_flashattn_attention.ipynb`) 의 에러는 커스텀 `MyFlashAttnBackend` 코드
안쪽에서 발생한 것으로 좁혀진다 — `flash_attn_attention_backend.py` 의 Metadata /
Builder / Impl 셋 중 하나.

반대로 이 셀이 실패하면 — 환경 자체가 문제. 점검 순서:
1. `vllm.__version__` 가 0.19.x 인가
2. CUDA / driver 버전이 vLLM 번들 FA2 와 호환되는가
3. `from vllm.vllm_flash_attn import flash_attn_varlen_func` 가 ImportError 없이 되는가